# Incompressible OpenFOAM Baseline: Lid-driven Cavity

## Preamble

In [1]:
# Standard Library
from pathlib import Path
import foamnordic as fno

### Directories

In [2]:
# Case Type and Name
CASE_TYPE = "laminar"
CASE_NAME = "lidDrivenCavity"
CATEGORY = "incompressible"

# FoamNordic Project Directory
PROJECT_DIR = Path.cwd().resolve()
BASE_DIR = next((path for path in (PROJECT_DIR, *PROJECT_DIR.parents) if (path / "tutorials" / "openfoam_tutorials").is_dir()), None)
if BASE_DIR is None:
    raise FileNotFoundError("Run this notebook from inside the FoamNordic source tree")
MAIN_DIR = BASE_DIR / "tutorials" / "foamnordic_tutorials" / CATEGORY
OF_SCRIPT_DIR = BASE_DIR / "tutorials" / "openfoam_tutorials" / CATEGORY / CASE_TYPE / CASE_NAME

# Output Directory
OUTPUT_DIR = MAIN_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### Configuration

In [3]:
# Local configuration
N_TASKS = 1

## Example - OpenFOAM on FoamNordic (Baseline)

### Case Definition

In [4]:
# Initialize the OpenFOAM case
case = fno.OpenFOAM.Case(
    name=CASE_NAME,
    case_dir=OF_SCRIPT_DIR,
    run_dir=OUTPUT_DIR,
    of_cmd="openfoam",
    shell="zsh",
    application="pimpleFoam",
    ranks=N_TASKS,
    integration=None,
)

case.initialize(ranks=N_TASKS, mesh="blockMesh", validate_mesh=True);

### Submit Job

In [5]:
# Connect the OpenFOAM case and execution resources
longship = fno.Longship(
    case=case,
    closures=(),
    transforms=(),
    observations=(),
    placement=fno.Attached(model_cpus_per_node=None, data_path="auto"),
    scheduler=None,
    name=None,
    verbose=True,
    numerics="convenient",
)

In [6]:
# Launch the workload
run = longship.launch(
    start_timeout=900.0,
    readiness_timeout=120.0,
    receive_memory_bytes=None,
    profile_every=0,
    termination_grace=30.0,
    orphan_timeout=30.0,
    verbose=None,
)

[FoamNordic] Preparing mesh with blockMesh: lidDrivenCavity
[FoamNordic] Mesh is ready: lidDrivenCavity
[FoamNordic] Sailing in background: lidDrivenCavity


In [7]:
# Wait for the job to complete (polling the job status)
result = run.stop(force=False, timeout=3600, progress=True)

In [10]:
# Summary of the job result
result.summary(style="compact", display=True);

Job ID,Name,Status,Partition,Node,Elapsed
-,lidDrivenCavity,succeeded,local,rc5183,00:00:37


### Post-processing

In [9]:
# Postprocessing
post = result.postprocess

velocity = post.field(name="U", time_idx=-1, physical_time=None)
pressure = post.field(name="p", time_idx=-1, physical_time=None)

print("U shape:", velocity.shape)
print("p shape:", pressure.shape)

statistics = post.statistics(
    fields=["U", "p"],
    time_idx=-1,
    physical_time=None,
    verbose=True,
)

U shape: (4096, 3)
p shape: (4096,)


Field,Min,Max,Mean,Std,RMS
U,8.131911e-06,8.736061e-01,1.765603e-01,1.458542e-01,2.290131e-01
p,-1.093240e-01,5.939280e-01,-2.095064e-02,3.247379e-02,3.864552e-02
